<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Leakage-Free Evaluation · Inference Only · Publication Prerequisite</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Sızıntısız İç Test</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">İç test AUC'si 0,998'den, eğitimle hiçbir kaynak görüntüyü paylaşmayan alt kümede kaça iniyor?</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Eğitim:</b> yok — kayıtlı kontrol noktaları</div>
    <div><b>Süre:</b> ~8 dakika</div>
    <div><b>Ölçüt:</b> ROC-AUC (sınıf oranından bağımsız)</div>
    <div><b>Ek:</b> sızıntılı ↔ temiz doğrudan karşılaştırma</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Gerekçe:</b> Bütünlük denetimi, kullanılan &ldquo;dengelenmiş&rdquo; kümede NORMAL sınıfının 1.575 kaynak görüntüden augmentasyonla 4.265 dosyaya çıkarıldığını ve test bölmesindeki NORMAL görüntülerin <b>%86,6'sının eğitimdeki bir görüntünün kopyası</b> olduğunu gösterdi. PNEUMONIA'da hiç kopya yok — yani sızıntı <b>sınıfa göre asimetrik.</b> Bu koşulda &ldquo;NORMAL'i ezberle, tanımadığına PNEUMONIA de&rdquo; stratejisi yüksek iç başarım üretir. Bu notebook, eğitimle hiçbir kaynak görüntüyü paylaşmayan alt kümede aynı modelleri yeniden değerlendirir.
  </div>
</div>

## Tanımlar: iki temizlik düzeyi

Bir havuz görüntüsünün &ldquo;temiz&rdquo; sayılması için eğitim setiyle hiçbir ortak
köken taşımaması gerekir. İki düzeyde tanımlanır:

| Düzey | Kural | Havuzda kalan (tekilleştirilmiş) |
|---|---|---|
| **Kaynak düzeyi** | `_aug_###` eki soyulduğunda elde edilen dosya kökü eğitimde yok | 85 NORMAL · 865 PNEUMONIA |
| **Hasta düzeyi** (katı) | Dosya adından çıkarılan hasta anahtarı eğitimde yok | 40 NORMAL · 138 PNEUMONIA |

Hasta düzeyi, tıbbi görüntülemede kabul gören ölçüttür ve **birincil** sonuç olarak
raporlanır; kaynak düzeyi daha geniş örneklem sunduğu için duyarlılık analizi olarak verilir.

Her iki durumda da aynı kökten gelen birden çok dosya varsa **yalnızca biri** tutulur
(augmente edilmemiş özgün sürüm tercih edilir); aksi halde test kümesi kendi içinde
tekrar eden görüntüler barındırır ve hatalar yapay olarak ilişkilenir.

### Sınıf dengesi neden zorlanmıyor

Temiz NORMAL sayısı azdır (85 / 40), PNEUMONIA ise boldur (865 / 138). **ROC-AUC sınıf
oranından bağımsız** olduğundan birincil ölçüt tüm temiz veri üzerinde hesaplanır.
Eşiğe bağlı ölçütler (doğruluk, F1, özgüllük) denge gerektirdiğinden, PNEUMONIA'dan
tekrarlı örnekleme (200 çekiliş) ile dengelenip ortalama ± standart sapma verilir —
tek bir çekilişin şansına bağlı kalınmaz.

### Doğrudan sızıntı etkisi ölçümü

Aynı pozitif küme (PNEUMONIA) sabit tutularak iki negatif küme karşılaştırılır:

$$\text{AUC}_{\text{sızıntılı}} = \text{AUC}\big(\text{NORMAL}_{\text{eğitim kopyası}} \cup \text{PNEUMONIA}\big)$$
$$\text{AUC}_{\text{temiz}} = \text{AUC}\big(\text{NORMAL}_{\text{temiz}} \cup \text{PNEUMONIA}\big)$$

Aradaki fark, doğrudan sızıntının başarım üzerindeki katkısıdır.

## Kurulum

| # | Sekme | Kimlik |
|---|---|---|
| 1 | Datasets | `yusufmurtaza01/chest-xray-pneumonia-balanced-dataset` |
| 2 | **Notebooks** | `segmentation-ablation-lung-focused-vit` (kontrol noktaları) |

GPU ve Internet açık olmalıdır (segmentasyon modeli indirilir).

In [ ]:
import os, re, io, gc, json, glob, time, random, zipfile, hashlib, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from collections import OrderedDict, Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models, datasets
from sklearn.metrics import (roc_curve, auc, average_precision_score, f1_score,
                             confusion_matrix, brier_score_loss)
from scipy import stats as sp_stats

SEED       = 42
N_BOOT     = 2000
N_BALANCE  = 200        # dengeli esik olcutleri icin tekrarli cekilis sayisi
ARMS       = ["raw", "roi", "lung"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ARM_LABEL = OrderedDict([("raw", "A · Segmentasyonsuz"),
                         ("roi", "B · Yalnizca RoI kirpma"),
                         ("lung", "C · Maske + RoI (onerilen)")])
ARM_COLOR = {"raw": "#B04A1E", "roi": "#C2900A", "lung": "#0D8FA2"}

plt.rcParams.update({
    "figure.dpi": 130, "figure.facecolor": "white", "savefig.facecolor": "white",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8A9499", "axes.labelcolor": "#1B2327", "text.color": "#1B2327",
    "xtick.color": "#5A686F", "ytick.color": "#5A686F",
    "grid.color": "#D8DFE1", "grid.linewidth": 0.7, "legend.frameon": False,
})
WORK = "/kaggle/working"
print(f"Cihaz: {device} | Torch {torch.__version__}")

In [ ]:
# ── Girdiler ─────────────────────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_classification_root(root=INPUT):
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0] if hits else None

BASE = find_classification_root()
assert BASE is not None, "Siniflandirma veri kumesi bulunamadi."

ckpts = {}
for p in glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True):
    b = os.path.basename(p).lower()
    for arm in ARMS:
        if f"_{arm}." in b:
            ckpts[arm] = p
missing = [a for a in ARMS if a not in ckpts]
assert not missing, f"Kontrol noktasi eksik: {missing}"

print("Veri kumesi:", BASE)
for a in ARMS:
    print(f"  ckpt {a:<5}: {ckpts[a]}")

In [ ]:
# ── Dosya envanteri + koken anahtarlari ──────────────────────────────────
rows = []
for split in ["train", "val", "test"]:
    for cls in ["NORMAL", "PNEUMONIA"]:
        d = os.path.join(BASE, split, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith((".jpeg", ".jpg", ".png")):
                rows.append({"split": split, "sinif": cls, "ad": f,
                             "yol": os.path.join(d, f)})
inv = pd.DataFrame(rows)

def src_key(name):
    return re.sub(r"_aug_\d+$", "", os.path.splitext(name)[0])

def patient_key(name):
    b = os.path.splitext(name)[0]
    m = re.match(r"(person\d+)_", b, re.I)
    if m:
        return "p_" + m.group(1).lower()
    m = re.match(r"(?:NORMAL\d*-)?IM-(\d+)", b, re.I)
    if m:
        return "im_" + m.group(1)
    return "x_" + b

inv["kaynak"] = inv["ad"].map(src_key)
inv["hasta"]  = inv["ad"].map(patient_key)
inv["aug"]    = inv["ad"].str.contains(r"_aug_\d+", regex=True)

print(f"Toplam {len(inv)} goruntu")
print(inv.groupby(["split", "sinif"]).size().unstack(fill_value=0).to_string())
print(f"\nNORMAL   : {inv[inv.sinif=='NORMAL'].kaynak.nunique()} kaynak -> "
      f"{(inv.sinif=='NORMAL').sum()} dosya  (augmente: {int(inv[inv.sinif=='NORMAL'].aug.sum())})")
print(f"PNEUMONIA: {inv[inv.sinif=='PNEUMONIA'].kaynak.nunique()} kaynak -> "
      f"{(inv.sinif=='PNEUMONIA').sum()} dosya  (augmente: {int(inv[inv.sinif=='PNEUMONIA'].aug.sum())})")

In [ ]:
# ── Ablasyondaki bolmeyi birebir yeniden uret + imza denetimi ────────────
EXPECTED_FP = {"train": "ccf23597ec992137", "val": "779ac7f6455a7ac8",
               "test": "3a25cbb40ab471d7"}

def file_sig(p):
    return f"{os.path.basename(p)}_{os.path.getsize(p) if os.path.exists(p) else 0}"

def fingerprint(recs):
    return hashlib.md5("|".join(sorted(r["ad"] for r in recs)).encode()).hexdigest()[:16]

random.seed(SEED)
tr_recs = inv[inv.split == "train"].to_dict("records")
pool = inv[inv.split.isin(["val", "test"])].copy()
tr_sigs = set(file_sig(r["yol"]) for r in tr_recs)
pool = pool[~pool["yol"].map(lambda p: file_sig(p) in tr_sigs)]

npool = pool[pool.sinif == "NORMAL"].to_dict("records")
ppool = pool[pool.sinif == "PNEUMONIA"].to_dict("records")
random.shuffle(npool); random.shuffle(ppool)
per = min(len(npool), len(ppool)) // 2
val_recs  = npool[:per] + ppool[:per]
test_recs = npool[per:2*per] + ppool[per:2*per]
random.shuffle(val_recs); random.shuffle(test_recs)

fps = {"train": fingerprint(tr_recs), "val": fingerprint(val_recs), "test": fingerprint(test_recs)}
print("Bolme imzalari:", fps)
ok = all(fps[k] == EXPECTED_FP[k] for k in fps)
print("Ablasyon kosumuyla ESLESTI." if ok else "UYARI: imzalar farkli!")
assert ok, "Bolme onceki kosumla eslesmiyor - ayni veri kumesi surumu ekli mi?"

TRAIN_SRC = set(r["kaynak"] for r in tr_recs)
TRAIN_PAT = set(r["hasta"] for r in tr_recs)

In [ ]:
# ── Degerlendirme alt kumelerinin tanimlanmasi ───────────────────────────
ev = pd.DataFrame(test_recs + val_recs)          # tum havuz (1728 goruntu)
ev["kaynak_temiz"] = ~ev["kaynak"].isin(TRAIN_SRC)
ev["hasta_temiz"]  = ~ev["hasta"].isin(TRAIN_PAT)
ev["orijinal_test"] = ev["ad"].isin({r["ad"] for r in test_recs})

def dedup(frame, key):
    '''Ayni kokten yalnizca bir dosya; augmente edilmemis olan tercih edilir.'''
    return frame.sort_values(["aug", "ad"]).drop_duplicates(key)

SETS = {}
SETS["orijinal_test"] = ev[ev.orijinal_test]
SETS["kaynak_temiz"]  = dedup(ev[ev.kaynak_temiz], "kaynak")
SETS["hasta_temiz"]   = dedup(ev[ev.hasta_temiz], "hasta")
SETS["sizintili_NORMAL"] = pd.concat([
    dedup(ev[(~ev.kaynak_temiz) & (ev.sinif == "NORMAL")], "ad"),
    dedup(ev[ev.sinif == "PNEUMONIA"], "kaynak")])

print("=" * 74)
print("  DEGERLENDIRME ALT KUMELERI")
print("=" * 74)
for k, v in SETS.items():
    c = v.groupby("sinif").size().to_dict()
    print(f"  {k:<20}: N={len(v):>4} | NORMAL={c.get('NORMAL',0):>4} "
          f"PNEUMONIA={c.get('PNEUMONIA',0):>4}")
print("=" * 74)

# Cikarim yapilacak benzersiz dosyalar
NEED = pd.concat(SETS.values()).drop_duplicates("yol").reset_index(drop=True)
print(f"\nCikarim yapilacak benzersiz goruntu: {len(NEED)}")

In [ ]:
# ── Modeller + segmentasyon ──────────────────────────────────────────────
def build_vit_inference(num_classes, dropout):
    m = models.vit_b_16(weights=None)
    in_f = m.heads.head.in_features
    m.heads.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_f, num_classes))
    return m

MODELS, CFG, CLASS_TO_IDX = {}, None, None
for arm in ARMS:
    ck = torch.load(ckpts[arm], map_location=device, weights_only=False)
    CFG, CLASS_TO_IDX = ck["config"], ck["class_to_idx"]
    m = build_vit_inference(CFG["num_classes"], CFG.get("dropout", 0.1)).to(device)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    MODELS[arm] = m.eval()
    print(f"  {arm:<5} yuklendi (Val F1 {ck.get('best_val_f1'):.4f})")
PNEU_IDX = CLASS_TO_IDX["PNEUMONIA"]
eval_tf = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["mean"], CFG["std"]),
])

import transformers
from transformers import AutoModel
print("\nianpan/chest-x-ray-basic yukleniyor...")
def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()
_of = getattr(transformers.modeling_utils.PreTrainedModel, "_finalize_model_loading", None)
try:
    if _of is not None:
        def _sf(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _of(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _sf
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _of is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _of
print("Segmentasyon modeli hazir.")

In [ ]:
# ── On-isleme: ablasyon notebook'u ile birebir ayni ──────────────────────
def load_image_any(path, short_max):
    pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size
    short = min(W0, H0)
    if short > short_max:
        s = short_max / short
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))

@torch.inference_mode()
def lung_mask_ianpan(gray_u8, out_hw):
    x = seg_model.preprocess(gray_u8)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    logits = seg_model(x)["mask"]
    logits = F.interpolate(logits, size=out_hw, mode="bilinear", align_corners=False)
    pred = logits.argmax(dim=1)[0].cpu().numpy()
    return ((pred == 1) | (pred == 2)).astype(np.uint8)

def prep_arms(rgb_u8, lung_u8, cfg):
    H, W = lung_u8.shape
    short = min(H, W); S = cfg["img_size"]
    out = {"raw": (cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA), None, True)}
    if lung_u8.sum() < 1:
        fb = cv2.resize(rgb_u8, (S, S), interpolation=cv2.INTER_AREA)
        out["roi"] = (fb, None, False); out["lung"] = (fb, None, False)
        return out
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung_u8, kern, iterations=1)
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0.0, 1.0)[..., None]
    fill = (np.array([m * 255.0 for m in cfg["mean"]], dtype=np.float32)
            if cfg["fill_mode"] == "mean" else np.zeros(3, dtype=np.float32))
    masked = (rgb_u8.astype(np.float32) * soft +
              fill[None, None, :] * (1.0 - soft)).clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max()); x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)
    bh, bw = (y1 - y0 + 1), (x1 - x0 + 1)
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side);     tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side);     tx0 = max(0, tx1 - side)
    out["roi"]  = (cv2.resize(rgb_u8[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA),
                   None, True)
    out["lung"] = (cv2.resize(masked[ty0:ty1, tx0:tx1], (S, S), interpolation=cv2.INTER_AREA),
                   None, True)
    return out

# ── Cikarim ──────────────────────────────────────────────────────────────
@torch.inference_mode()
def infer_all(frame):
    probs = {a: [] for a in ARMS}
    t0 = time.time()
    for i, r in enumerate(frame.itertuples()):
        rgb, gray = load_image_any(r.yol, CFG["orig_short_max"])
        lung = lung_mask_ianpan(gray, gray.shape)
        arms = prep_arms(rgb, lung, CFG)
        for a in ARMS:
            x = eval_tf(Image.fromarray(arms[a][0])).unsqueeze(0).to(device)
            probs[a].append(torch.softmax(MODELS[a](x), 1)[0, PNEU_IDX].item())
        if (i + 1) % 300 == 0:
            print(f"  {i+1}/{len(frame)} ({time.time()-t0:.0f} sn)")
    print(f"  bitti: {len(frame)} goruntu, {time.time()-t0:.0f} sn")
    return probs

print(f"Cikarim basliyor ({len(NEED)} goruntu x 3 kol)...")
P = infer_all(NEED)
for a in ARMS:
    NEED[f"p_{a}"] = P[a]
NEED["y"] = (NEED["sinif"] == "PNEUMONIA").astype(int)
PROB = NEED.set_index("yol")

In [ ]:
# ── Istatistik araclari ──────────────────────────────────────────────────
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1); i = j
    T2 = np.empty(N, dtype=float); T2[J] = T + 1
    return T2

def _fast_delong(preds_sorted, m):
    n = preds_sorted.shape[1] - m
    pos, neg = preds_sorted[:, :m], preds_sorted[:, m:]
    k = preds_sorted.shape[0]
    tx = np.empty([k, m]); ty = np.empty([k, n]); tz = np.empty([k, m + n])
    for r in range(k):
        tx[r, :] = _midrank(pos[r, :]); ty[r, :] = _midrank(neg[r, :])
        tz[r, :] = _midrank(preds_sorted[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.cov(v01); sy = np.cov(v10)
    if k == 1:
        sx = np.array([[float(sx)]]); sy = np.array([[float(sy)]])
    return aucs, sx / m + sy / n

def delong_test(y_true, p1, p2):
    y = np.asarray(y_true).astype(int)
    order = np.argsort(-y, kind="mergesort"); m = int(y.sum())
    preds = np.vstack((np.asarray(p1), np.asarray(p2)))[:, order]
    aucs, cov = _fast_delong(preds, m)
    l = np.array([[1.0, -1.0]]); var = float(l.dot(cov).dot(l.T))
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = float((aucs[0] - aucs[1]) / np.sqrt(var))
    return aucs[0], aucs[1], z, float(2 * (1 - sp_stats.norm.cdf(abs(z))))

def boot_auc(y, p, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p)
    ip, ineg = np.where(y == 1)[0], np.where(y == 0)[0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        ii = np.concatenate([rng.choice(ip, len(ip), True), rng.choice(ineg, len(ineg), True)])
        fpr, tpr, _ = roc_curve(y[ii], p[ii]); out[b] = auc(fpr, tpr)
    return out

def balanced_threshold_metrics(y, p, n_draw=N_BALANCE, seed=SEED, thr=0.5):
    '''Cok olan sinifi tekrarli ornekleyerek dengeler; ortalama +/- ss doner.'''
    rng = np.random.default_rng(seed)
    y = np.asarray(y); p = np.asarray(p)
    ip, ineg = np.where(y == 1)[0], np.where(y == 0)[0]
    k = min(len(ip), len(ineg))
    acc, f1s, sens, spec = [], [], [], []
    for _ in range(n_draw):
        ii = np.concatenate([rng.choice(ip, k, False), rng.choice(ineg, k, False)])
        yy, pp = y[ii], (p[ii] >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(yy, pp, labels=[0, 1]).ravel()
        se = tp / (tp + fn + 1e-9); sp = tn / (tn + fp + 1e-9)
        pr = tp / (tp + fp + 1e-9)
        acc.append((pp == yy).mean()); f1s.append(2 * pr * se / (pr + se + 1e-9))
        sens.append(se); spec.append(sp)
    f = lambda v: (float(np.mean(v)), float(np.std(v)))
    return {"Acc": f(acc), "F1": f(f1s), "Duyarlilik": f(sens), "Ozgulluk": f(spec), "n_sinif": k}

print("Istatistik araclari hazir.")

In [ ]:
# ── Ana tablo: alt kume x kol ────────────────────────────────────────────
SET_LABEL = {"orijinal_test": "Orijinal test (sizintili)",
             "sizintili_NORMAL": "Yalnizca sizintili NORMAL",
             "kaynak_temiz": "Kaynak duzeyi temiz",
             "hasta_temiz": "Hasta duzeyi temiz (birincil)"}
ORDER = ["orijinal_test", "sizintili_NORMAL", "kaynak_temiz", "hasta_temiz"]

rows, CI = [], {}
for key in ORDER:
    sub = SETS[key]
    y = (sub["sinif"] == "PNEUMONIA").astype(int).values
    for a in ARMS:
        p = PROB.loc[sub["yol"], f"p_{a}"].values
        fpr, tpr, _ = roc_curve(y, p); A = auc(fpr, tpr)
        bb = boot_auc(y, p)
        CI[(key, a)] = (float(np.percentile(bb, 2.5)), float(np.percentile(bb, 97.5)))
        bm = balanced_threshold_metrics(y, p)
        rows.append({"alt kume": SET_LABEL[key], "kol": a, "N": len(y),
                     "NORMAL": int((y == 0).sum()), "PNEU": int((y == 1).sum()),
                     "AUC": A, "GA alt": CI[(key, a)][0], "GA ust": CI[(key, a)][1],
                     "AP": average_precision_score(y, p),
                     "Acc (dengeli)": bm["Acc"][0], "F1 (dengeli)": bm["F1"][0],
                     "Duyarlilik": bm["Duyarlilik"][0], "Ozgulluk": bm["Ozgulluk"][0]})
df_res = pd.DataFrame(rows)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
pd.set_option("display.width", 200)
print("=" * 132)
print("  SIZINTILI vs TEMIZ IC TEST")
print("=" * 132)
print(df_res.to_string(index=False))
print("=" * 132)
print("  Esige bagli olcutler, cok olan sinifin tekrarli ornekleme ile dengelenmesiyle")
print(f"  hesaplandi ({N_BALANCE} cekilis ortalamasi, esik = 0,50).")

In [ ]:
# ── Sizinti etkisinin dogrudan olcumu (ayni pozitifler) ──────────────────
print("=" * 100)
print("  SIZINTININ KATKISI  (pozitif kume sabit; yalnizca negatif kume degisiyor)")
print("=" * 100)
rows = []
pos = dedup(ev[ev.sinif == "PNEUMONIA"], "kaynak")
neg_leak = dedup(ev[(~ev.kaynak_temiz) & (ev.sinif == "NORMAL")], "ad")
neg_clean = dedup(ev[ev.hasta_temiz & (ev.sinif == "NORMAL")], "hasta")
for a in ARMS:
    pp = PROB.loc[pos["yol"], f"p_{a}"].values
    yl = np.r_[np.ones(len(pos)), np.zeros(len(neg_leak))]
    pl = np.r_[pp, PROB.loc[neg_leak["yol"], f"p_{a}"].values]
    yc = np.r_[np.ones(len(pos)), np.zeros(len(neg_clean))]
    pc = np.r_[pp, PROB.loc[neg_clean["yol"], f"p_{a}"].values]
    f1_, t1_, _ = roc_curve(yl, pl); A_leak = auc(f1_, t1_)
    f2_, t2_, _ = roc_curve(yc, pc); A_clean = auc(f2_, t2_)
    bl = boot_auc(yl, pl, 1000); bc = boot_auc(yc, pc, 1000)
    rows.append({"kol": a, "AUC sizintili NORMAL": A_leak, "AUC temiz NORMAL": A_clean,
                 "fark": A_leak - A_clean,
                 "temiz GA": f"[{np.percentile(bc,2.5):.3f}, {np.percentile(bc,97.5):.3f}]",
                 "n NORMAL sizintili": len(neg_leak), "n NORMAL temiz": len(neg_clean)})
df_leak = pd.DataFrame(rows)
print(df_leak.to_string(index=False))
print("=" * 100)

# Olasilik dagilimlari: ezberleme izi
print("\n  NORMAL goruntulerde P(pnomoni) ortalamasi")
print(f"  {'kol':<6}{'sizintili':>12}{'temiz':>10}{'MWU p':>12}")
for a in ARMS:
    vl = PROB.loc[neg_leak["yol"], f"p_{a}"].values
    vc = PROB.loc[neg_clean["yol"], f"p_{a}"].values
    _, pw = sp_stats.mannwhitneyu(vl, vc, alternative="two-sided")
    print(f"  {a:<6}{vl.mean():>12.4f}{vc.mean():>10.4f}{pw:>12.2e}")

In [ ]:
# ── Figur 1: alt kume x kol AUC ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10.2, 4.8))
xs = np.arange(len(ORDER)); w = 0.26
for k, a in enumerate(ARMS):
    vals = np.array([df_res[(df_res["alt kume"] == SET_LABEL[s]) & (df_res.kol == a)]["AUC"].iloc[0]
                     for s in ORDER])
    lo = np.array([CI[(s, a)][0] for s in ORDER]); hi = np.array([CI[(s, a)][1] for s in ORDER])
    pos_ = xs + (k - 1) * w
    ax.bar(pos_, vals, width=w * 0.88, color=ARM_COLOR[a], edgecolor="white",
           linewidth=1.6, label=ARM_LABEL[a], zorder=3)
    ax.errorbar(pos_, vals, yerr=[vals - lo, hi - vals], fmt="none",
                ecolor="#2B3438", elinewidth=1.1, capsize=3.5, zorder=4)
    for x, v, h in zip(pos_, vals, hi):
        ax.text(x, h + 0.012, f"{v:.3f}", ha="center", va="bottom", fontsize=8.4, zorder=5)
ax.axhline(0.5, color="#8A9499", ls=":", lw=1, zorder=2)
ax.set_xticks(xs)
ax.set_xticklabels([SET_LABEL[s].replace(" (", "\n(") for s in ORDER], fontsize=9)
ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.4, 1.08)
ax.yaxis.grid(True, alpha=.55); ax.set_axisbelow(True)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=3, fontsize=9)
fig.suptitle("Ic test basarimi, sizinti giderildiginde ne oluyor?  (%95 bootstrap GA)",
             fontsize=12, fontweight="bold", y=1.10)
plt.tight_layout()
plt.savefig(os.path.join(WORK, "clean_fig_01_auc.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figur 2: NORMAL goruntulerde olasilik dagilimi (ezberleme izi) ───────
fig, axes = plt.subplots(1, len(ARMS), figsize=(4.4 * len(ARMS), 3.9), squeeze=False)
bins = np.linspace(0, 1, 26)
for ax, a in zip(axes[0], ARMS):
    vl = PROB.loc[neg_leak["yol"], f"p_{a}"].values
    vc = PROB.loc[neg_clean["yol"], f"p_{a}"].values
    ax.hist(vl, bins=bins, density=True, alpha=.6, color="#8A9499",
            label=f"sizintili (n={len(vl)})", edgecolor="white", linewidth=.6)
    ax.hist(vc, bins=bins, density=True, alpha=.75, color=ARM_COLOR[a],
            label=f"temiz (n={len(vc)})", edgecolor="white", linewidth=.6)
    ax.axvline(0.5, color="#5A686F", ls=":", lw=1)
    ax.set_title(ARM_LABEL[a], color=ARM_COLOR[a], fontsize=10)
    ax.set_xlabel("P(pnomoni)"); ax.set_ylabel("yogunluk")
    ax.legend(fontsize=8); ax.grid(alpha=.4); ax.set_axisbelow(True)
fig.suptitle("NORMAL goruntulerde tahmin dagilimi — saga kayma, temiz orneklerde hata demektir",
             fontsize=11.5, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(WORK, "clean_fig_02_prob_dist.png"), dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ── Kayit ────────────────────────────────────────────────────────────────
df_res.to_csv(os.path.join(WORK, "clean_internal_metrics.csv"), index=False)
df_leak.to_csv(os.path.join(WORK, "clean_leakage_effect.csv"), index=False)
NEED.drop(columns=["yol"]).to_csv(os.path.join(WORK, "clean_per_image.csv"), index=False)

ozet = {
    "bolme_imzalari": fps,
    "alt_kume_boyutlari": {k: {"N": int(len(v)),
                               **{c: int(n) for c, n in v.groupby("sinif").size().items()}}
                           for k, v in SETS.items()},
    "auc": {f"{k}__{a}": float(df_res[(df_res['alt kume'] == SET_LABEL[k]) &
                                      (df_res.kol == a)]["AUC"].iloc[0])
            for k in ORDER for a in ARMS},
    "auc_ci": {f"{k}__{a}": list(CI[(k, a)]) for k in ORDER for a in ARMS},
}
with open(os.path.join(WORK, "clean_internal_summary.json"), "w", encoding="utf-8") as f:
    json.dump(ozet, f, indent=2, ensure_ascii=False)

print("=" * 78)
print("  OZET — hasta duzeyi temiz alt kume (birincil sonuc)")
print("=" * 78)
for a in ARMS:
    o = df_res[(df_res["alt kume"] == SET_LABEL["orijinal_test"]) & (df_res.kol == a)]["AUC"].iloc[0]
    c = df_res[(df_res["alt kume"] == SET_LABEL["hasta_temiz"]) & (df_res.kol == a)]["AUC"].iloc[0]
    lo, hi = CI[("hasta_temiz", a)]
    print(f"  {a:<6} orijinal {o:.4f}  ->  temiz {c:.4f}  GA [{lo:.3f}, {hi:.3f}]  "
          f"dusus {o-c:+.4f}")
print("=" * 78)
for f in sorted(os.listdir(WORK)):
    if f.startswith("clean"):
        print(f"  {f:<34} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")

## Sonucun okunması

**Beklenen:** orijinal (sızıntılı) alt kümede AUC ≈ 0,998; hasta düzeyi temiz alt kümede
belirgin biçimde düşük. Aradaki fark, iç test başarımının ne kadarının ezberlemeden
geldiğini gösterir.

**Olasılık dağılımı figürü** doğrudan kanıttır: sızıntılı NORMAL görüntüler 0'a yapışıksa
ve temiz NORMAL görüntüler sağa yayılıyorsa, model o görüntüleri *tanıyor*, sınıflandırmıyor.

**Düşüş küçük çıkarsa** sızıntı başarımı şişirmemiş demektir — bu da raporlanabilir ve
iç test sayısını kurtarır.

### Kısıtlar (makalede belirtilmeli)

1. **Örneklem küçük.** Hasta düzeyi temiz alt kümede sınıf başına ~40 görüntü kalır;
   güven aralıkları geniştir. Bu, veri kümesinin yapısından kaynaklanır, tasarım
   tercihi değildir.
2. **Modeller yine de sızıntılı veriyle eğitilmiştir.** Bu ölçüm, temiz bir *test*
   sağlar; temiz bir *eğitim* değil. NORMAL sınıfının efektif büyüklüğü 3.400 değil,
   yaklaşık 1.244 özgün görüntüdür.
3. **Hasta anahtarı dosya adından çıkarılmıştır.** `IM-####` kalıbının Kermany
   kümesinde çalışma/hasta düzeyinde olduğu varsayılmıştır; gerçek hasta kimlikleri
   yayımlanmadığından bu bir üst sınır tahminidir.

> Bu üç kısıt, sonucu geçersiz kılmaz; aksine iç test sayısının neden **birincil kanıt
> olarak kullanılmaması** gerektiğini gösterir. Çalışmanın tüm ana iddiaları bağımsız
> dış kümelerde (RSNA, NIH) ölçülmüştür ve bu sızıntıdan etkilenmez.